### Collisional-radiative model for Argon Plasma
#### Rates merging
See: https://doi.org/10.1063/1.3585688

In [1]:
import numpy as np
import tensorflow as tf

from silx.io.dictdump import dicttoh5, h5todict

Inputs

In [2]:
path_to_kin = "./../rates/kin_fit.hdf5"
path_to_rad = "./../rates/rad_fit.hdf5"
path_to_rates = "./../rates/rates.hdf5"

Loading rates databases

In [3]:
kin = h5todict(path_to_kin)
rad = h5todict(path_to_rad)

Cleaning rates databases
> Kinetics

In [4]:
_ = rad.pop("T")
_ = rad.pop("BFp")
_ = rad.pop("FF")
_ = rad["BB"].pop("plato_id")
_ = rad["BF"].pop("plato_id")
_ = rad["BB"].pop("equation")
_ = rad["BF"].pop("equation")

> Radiation

In [5]:
_ = kin.pop("T")
_ = kin.pop("EN")
_ = kin.pop("EI")
for k in ("EXe","EXh","Ie","Ih"):
  _ = kin[k].pop("plato_id")
  _ = kin[k].pop("equation")

Merging rates databases

In [6]:
rates_raw = tf.nest.map_structure(np.squeeze, {**kin, **rad})
rates_raw["BB"]["values"] = {
  "A": rates_raw["BB"]["values"],
  "lambda": rates_raw["BB"].pop("lambda")
}
rates_raw["BF"]["values"]["lambda"] = rates_raw["BF"].pop("lambda")

In [7]:
rates = {}
nb_comp = 33
for k1 in ("EXe", "EXh", "BB"):
  rates[k1] = {}
  for k2, c2 in rates_raw[k1]["values"].items():
    rates[k1][k2] = np.zeros((nb_comp, nb_comp))
    rates[k1][k2][:31,:31] = c2
for k1 in ("Ie", "Ih", "BF"):
  rates[k1] = {}
  for k2, c2 in rates_raw[k1]["values"].items():
    rates[k1][k2] = np.zeros((nb_comp, nb_comp))
    rates[k1][k2][:31,-2:] = c2

In [8]:
dicttoh5(rates, path_to_rates)